In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS catalog.silver;

In [0]:
%sql

SELECT * FROM catalog.bronze.hospitals_raw;

hospital_id,hospital_name,branch_code,city,region,hospital_class,bed_count,year_opened,status,network_region_cluster,_rescued_data,_source_file,_ingest_timestamp,ingestion_date
H001,MediLuzon General Hospital - Manila Flagship,MLH-MNL,Manila,NCR,Tertiary,350,2005,Active,NCR Cluster,null,abfss://data@datastoragea.dfs.core.windows.net/staging/hospitals_raw/hospitals_raw,2026-05-05T11:36:18.213Z,2026-05-05
H002,MediLuzon General Hospital - Quezon City,MLH-QC,Quezon City,NCR,Tertiary,280,2008,Active,NCR Cluster,null,abfss://data@datastoragea.dfs.core.windows.net/staging/hospitals_raw/hospitals_raw,2026-05-05T11:36:18.213Z,2026-05-05
H003,MediLuzon Specialist Center - Makati,MLH-MKT,Makati,NCR,Specialty,180,2010,Active,NCR Cluster,null,abfss://data@datastoragea.dfs.core.windows.net/staging/hospitals_raw/hospitals_raw,2026-05-05T11:36:18.213Z,2026-05-05
H004,MediLuzon Community Hospital - Batangas,MLH-BTG,Batangas City,Region IV-A,Secondary,120,2012,Active,Luzon Cluster,null,abfss://data@datastoragea.dfs.core.windows.net/staging/hospitals_raw/hospitals_raw,2026-05-05T11:36:18.213Z,2026-05-05
H005,MediLuzon Community Hospital - Pampanga,MLH-PMP,"San Fernando, Pampanga",Region III,Secondary,150,2013,Active,Luzon Cluster,null,abfss://data@datastoragea.dfs.core.windows.net/staging/hospitals_raw/hospitals_raw,2026-05-05T11:36:18.213Z,2026-05-05
H006,MediLuzon General Hospital - Cebu,MLH-CBU,Cebu City,Region VII,Tertiary,220,2015,Active,Visayas Cluster,null,abfss://data@datastoragea.dfs.core.windows.net/staging/hospitals_raw/hospitals_raw,2026-05-05T11:36:18.213Z,2026-05-05
H007,MediLuzon Community Hospital - Iloilo,MLH-ILO,Iloilo City,Region VI,Secondary,130,2016,Active,Visayas Cluster,null,abfss://data@datastoragea.dfs.core.windows.net/staging/hospitals_raw/hospitals_raw,2026-05-05T11:36:18.213Z,2026-05-05
H008,MediLuzon General Hospital - Davao,MLH-DVO,Davao City,Region XI,Tertiary,200,2017,Active,Mindanao Cluster,null,abfss://data@datastoragea.dfs.core.windows.net/staging/hospitals_raw/hospitals_raw,2026-05-05T11:36:18.213Z,2026-05-05
H009,MediLuzon Community Hospital - Cagayan de Oro,MLH-CDO,Cagayan de Oro,Region X,Secondary,110,2019,Active,Mindanao Cluster,null,abfss://data@datastoragea.dfs.core.windows.net/staging/hospitals_raw/hospitals_raw,2026-05-05T11:36:18.213Z,2026-05-05
H010,MediLuzon Specialist Center - Bonifacio Global City,MLH-BGC,Taguig,NCR,Specialty,160,2021,Active,NCR Cluster,null,abfss://data@datastoragea.dfs.core.windows.net/staging/hospitals_raw/hospitals_raw,2026-05-05T11:36:18.213Z,2026-05-05


In [0]:
from pyspark.sql.functions import current_timestamp
from delta.tables import DeltaTable

In [0]:
bronze_table = 'catalog.bronze.hospitals_raw'
silver_table = 'catalog.silver.dim_hospitals'
checkpoint_path = "abfss://data@datastoragea.dfs.core.windows.net/silver/dim_hospitals/checkpoint/"

df_hospitals_bronze = spark.readStream.table(bronze_table)

df_hospitals_clean = (
    df_hospitals_bronze
        .drop(
            "status",
            "_rescued_data",
            "_source_file",
            "_ingest_timestamp",
            "ingestion_date"
        )
        .withColumn("load_timestamp", current_timestamp())
)

def merge_dim_hospitals(batch_df, batch_id):
    batch_df = batch_df.drop("status").dropDuplicates(["hospital_id"])

    if not spark.catalog.tableExists(silver_table):
        # First run — create the table
        (batch_df.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(silver_table))  
        return
    
    # Load Delta table by name and upsert into Silver
    dim_hospitals = DeltaTable.forName(spark, silver_table)

    (dim_hospitals.alias("t")
        .merge(
            batch_df.alias("s"),
            "t.hospital_id = s.hospital_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())
    

In [0]:
(   
    df_hospitals_clean.writeStream
        .foreachBatch(merge_dim_hospitals)  # for every batch, merge_dim_hospitals is run
        .outputMode("update")
        .trigger(availableNow=True)
        .option("checkpointLocation", checkpoint_path)
        .start()
)

In [0]:
%sql
SELECT * FROM catalog.silver.dim_hospitals;

hospital_id,hospital_name,branch_code,city,region,hospital_class,bed_count,year_opened,network_region_cluster,load_timestamp
H005,MediLuzon Community Hospital - Pampanga,MLH-PMP,"San Fernando, Pampanga",Region III,Secondary,150,2013,Luzon Cluster,2026-05-21T14:37:27.725Z
H009,MediLuzon Community Hospital - Cagayan de Oro,MLH-CDO,Cagayan de Oro,Region X,Secondary,110,2019,Mindanao Cluster,2026-05-21T14:37:27.725Z
H004,MediLuzon Community Hospital - Batangas,MLH-BTG,Batangas City,Region IV-A,Secondary,120,2012,Luzon Cluster,2026-05-21T14:37:27.725Z
H007,MediLuzon Community Hospital - Iloilo,MLH-ILO,Iloilo City,Region VI,Secondary,130,2016,Visayas Cluster,2026-05-21T14:37:27.725Z
H010,MediLuzon Specialist Center - Bonifacio Global City,MLH-BGC,Taguig,NCR,Specialty,160,2021,NCR Cluster,2026-05-21T14:37:27.725Z
H008,MediLuzon General Hospital - Davao,MLH-DVO,Davao City,Region XI,Tertiary,200,2017,Mindanao Cluster,2026-05-21T14:37:27.725Z
H006,MediLuzon General Hospital - Cebu,MLH-CBU,Cebu City,Region VII,Tertiary,220,2015,Visayas Cluster,2026-05-21T14:37:27.725Z
H002,MediLuzon General Hospital - Quezon City,MLH-QC,Quezon City,NCR,Tertiary,280,2008,NCR Cluster,2026-05-21T14:37:27.725Z
H001,MediLuzon General Hospital - Manila Flagship,MLH-MNL,Manila,NCR,Tertiary,350,2005,NCR Cluster,2026-05-21T14:37:27.725Z
H003,MediLuzon Specialist Center - Makati,MLH-MKT,Makati,NCR,Specialty,180,2010,NCR Cluster,2026-05-21T14:37:27.725Z


In [0]:
# Data quality check
from pyspark.sql.functions import col, count, min, max

print("DIM HOSPITALS")
df= spark.read.table("catalog.silver.dim_hospitals")
total = df.count()
print(f"Total rows: {total} (expected: 10)")

dup_id = total - df.select("hospital_id").distinct().count()
print(f"Duplicate hospital_ids: {'OK' if dup_id == 0 else dup_id}")

for c in ["hospital_id", "hospital_name", "hospital_class", "bed_count",
          "network_region_cluster"]:
    n = df.filter(col(c).isNull()).count()
    print(f"  {'OK' if n == 0 else 'CHECK'} {c}: {n} nulls")

print("Hospital class distribution:")
df.groupBy("hospital_class").count().show()

print("Cluster distribution:")
df.groupBy("network_region_cluster").count().show()

df.select(
    min("bed_count").alias("min_beds"),
    max("bed_count").alias("max_beds"),
    min("year_opened").alias("earliest"),
    max("year_opened").alias("latest")
).show()


DIM HOSPITALS
Total rows: 10 (expected: 10)
Duplicate hospital_ids: OK
  OK hospital_id: 0 nulls
  OK hospital_name: 0 nulls
  OK hospital_class: 0 nulls
  OK bed_count: 0 nulls
  OK network_region_cluster: 0 nulls
Hospital class distribution:
+--------------+-----+
|hospital_class|count|
+--------------+-----+
|     Secondary|    4|
|      Tertiary|    4|
|     Specialty|    2|
+--------------+-----+

Cluster distribution:
+----------------------+-----+
|network_region_cluster|count|
+----------------------+-----+
|      Mindanao Cluster|    2|
|         Luzon Cluster|    2|
|       Visayas Cluster|    2|
|           NCR Cluster|    4|
+----------------------+-----+

+--------+--------+--------+------+
|min_beds|max_beds|earliest|latest|
+--------+--------+--------+------+
|     110|     350|    2005|  2021|
+--------+--------+--------+------+

